# 功能一、图片识别
+ 上传食材图片，自动识别其实的食材（要用到多模态）

In [4]:
import base64
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# ========== 1. 加载环境变量 ==========
# 请确保 .env 文件里已经配置了 DASHSCOPE_BASE_URL, DASHSCOPE_API_KEY 和 DEEPSEEK_API_KEY
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)


# ========== 2. 封装图片识别类 ==========
class FoodVisionAgent:
    def __init__(self):
        """初始化时，创建文字模型和视觉模型，并分别创建对应的 Agent"""
        
        # ② 负责文字交流的 Qwen 模型
        self.qwen_model = init_chat_model(
            model="qwen3.7-flash-2026-07-15",
            model_provider="openai",
            base_url=os.getenv("DASHSCOPE_BASE_URL"),
            api_key=os.getenv("DASHSCOPE_API_KEY")
        )
        self.text_agent = create_agent(model=self.qwen_model, tools=[])

        # ② 负责图片识别的 DeepSeek 模型
        # ⚠️ 注意：如果这个模型报错（提示不支持视觉或多模态），请将 model 换成 "qwen-vl-plus" 并换成 Qwen 的 Key 和 Base URL。
        self.vision_model = init_chat_model(
            model="deepseek-v4-flash-0731",
            api_key=os.getenv("DEEPSEEK_API_KEY")
        )
        self.vision_agent = create_agent(model=self.vision_model, tools=[])

    def _encode_image(self, image_path: str) -> str:
        """私有方法：将本地图片转换为 Base64 编码字符串"""
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"❌ 找不到本地图片，请检查路径：{image_path}")
            
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")

    def analyze_food(self, image_source: str, prompt: str = "请仔细识别这张图片中有哪些食材，并以列表形式输出。"):
        """
        对外暴露的核心接口：传入图片路径或URL，自动识别食材并流式输出结果。
        """
        content_list = [{"type": "text", "text": prompt}]

        # 判断图片是网络 URL 还是本地路径
        if image_source.startswith(("http://", "https://")):
            # 在线图片
            content_list.append({
                "type": "image_url",
                "image_url": {"url": image_source}
            })
            print(f"🌐 正在识别在线图片...")
        else:
            # 本地图片
            mime_type = "image/png" if image_source.lower().endswith(".png") else "image/jpeg"
            base64_data = self._encode_image(image_source)
            data_url = f"data:{mime_type};base64,{base64_data}"
            
            content_list.append({
                "type": "image_url",
                "image_url": {"url": data_url}
            })
            print(f"💻 正在识别本地图片: {image_source}")

        # 构建多模态消息
        msg = HumanMessage(content=content_list)

        # 调用视觉模型 Agent 进行流式输出
        print("🤖 AI 识别结果：\n" + "-"*30)
        try:
            stream = self.vision_agent.stream(
                {"messages": [msg]},
                stream_mode="messages"
            )
            for chunk, metadata in stream:
                if chunk.content:
                    print(chunk.content, end="", flush=True)
        except Exception as e:
            print(f"\n❌ 识别过程中发生错误: {e}")
            print("提示：请确认你的 deepseek-v4-flash-0731 模型支持多模态视觉，如果不支持，请换用 qwen-vl-plus 等视觉模型。")
        print("\n" + "-"*30)


# ========== 3. 使用示例 ==========
if __name__ == "__main__":
    # 实例化识别器
    analyzer = FoodVisionAgent()

    # 测试用例 1：识别在线图片
    online_image = "https://ts3.tc.mm.bing.net/th/id/OIP-C.YUmiTYnc4_RS1FBNeuIaKQHaE7?r=0&rs=1&pid=ImgDetMain"
    analyzer.analyze_food(online_image)

    # 测试用例 2：识别本地图片（取消下面代码的注释即可测试）
    # local_image = r"E:\我的电脑桌面\图图图\2.png"
    # analyzer.analyze_food(local_image)

🌐 正在识别在线图片...
🤖 AI 识别结果：
------------------------------

❌ 识别过程中发生错误: Error code: 400 - {'error': {'message': 'The supported API model names are deepseek-flash, deepseek-v4-pro, but you passed deepseek-v4-flash-0731. (request_id: 76dfbf1f-43e7-4869-b56b-3d266d4cc722)', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
提示：请确认你的 deepseek-v4-flash-0731 模型支持多模态视觉，如果不支持，请换用 qwen-vl-plus 等视觉模型。

------------------------------
